# Adaptive step-size control

A fixed grid spends the same effort everywhere. An adaptive solver chooses its own
steps instead: long ones where the solution is smooth, short ones where it changes
quickly. After every trial step it estimates the local error, from quantities the
filter step computes anyway, and compares it with a tolerance. If the error is too
large, the step is retried with a smaller step size $h$; otherwise it is accepted
and a *controller* proposes the next $h$.

In [ ]:
import logging

import jax
import jax.numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from tueplots import bundles, cycler
from tueplots.constants.color import palettes

from ode_filters import (
    IWP,
    ODEInformation,
    PController,
    gaussian_filter_adaptive,
    rts_smoother,
    taylor_mode_initialization,
)

jax.config.update("jax_enable_x64", True)

plt.rcParams.update(bundles.tue_ai_beamer())
plt.rcParams.update(cycler.cycler(color=palettes.paultol_bright))
set_matplotlib_formats("svg")  # vector figures stay sharp at any size
# Fall back to the default font quietly where Roboto Condensed is not installed.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
REFERENCE = palette[6]  # grey: exact solutions and other references

## Basic usage

`gaussian_filter_adaptive` takes the times you want the solution at, `save_at`, and
steps adaptively in between. It returns a `FilterResult` on `save_at`, as
`gaussian_filter` does on its grid, and the whole solve can be `jit`-compiled,
`vmap`-ed and differentiated.

As an example, take a solution that decays slowly and is hit by a short pulse at
$t = 5$:

$$\dot x = -0.3\,x + 5\,e^{-((t - 5)/0.15)^2}, \qquad x(0) = 1, \qquad t \in [0, 10].$$

In [ ]:
def vf(x, *, t):
    return -0.3 * x + 5.0 * np.exp(-(((t - 5.0) / 0.15) ** 2))


prior = IWP(q=2, d=1)
mu_0, P0_sqr = taylor_mode_initialization(vf, np.array([1.0]), q=2)
measure = ODEInformation(vf, prior.E0, prior.E1)

save_at = np.linspace(0.0, 10.0, 101)
result = gaussian_filter_adaptive(
    mu_0, P0_sqr, prior, measure, save_at, atol=1e-4, rtol=1e-3
)
print("reached every save time:", bool(result.success))

## Where the steps go

The save grid hides the steps the solver actually took; `record_steps=` records
them. In the figure, each dot on the solution (top) is an accepted step, and the
bottom panel shows the size of each step.

In [ ]:
# record_steps keeps every accepted step, not only the save grid.
run = gaussian_filter_adaptive(
    mu_0, P0_sqr, prior, measure, np.array([0.0, 10.0]),
    atol=1e-4, rtol=1e-3, record_steps=1000,
)
n = int(run.steps.count)
t_steps = np.concatenate([np.zeros(1), run.steps.t[:n]])
x_steps = np.concatenate([mu_0[:1], run.steps.m[:n, 0]])
h = run.steps.h[:n]
t_dense = np.linspace(0.0, 10.0, 401)
x_dense = gaussian_filter_adaptive(
    mu_0, P0_sqr, prior, measure, t_dense, atol=1e-4, rtol=1e-3
).m[:, 0]

print(
    f"{n} accepted steps ({int(run.n_rejected.sum())} rejected), from h = {h.min():.3f} "
    f"to h = {h.max():.2f}. A fixed grid with the smallest step would need "
    f"{int(np.ceil(10.0 / h.min()))}."
)

fig, (ax_x, ax_h) = plt.subplots(2, 1, sharex=True)
ax_x.plot(t_dense, x_dense, color=REFERENCE, lw=2.5, label="solution")
ax_x.plot(t_steps, x_steps, "o", ms=2.5, color=palette[0], label="accepted steps")
ax_x.set_ylabel("$x$")
ax_h.semilogy(t_steps[1:], h, "o-", ms=2, color=palette[0])
ax_h.set_ylabel("step size $h$")
ax_h.set_xlabel("$t$")
ax_x.legend(loc="lower left")
plt.show()


Away from the pulse the steps are about half a time unit long. As the pulse
arrives the error estimate grows, a few steps are rejected and retried shorter, and
through the pulse the steps are about thirty times shorter; once it has passed they
grow back. By default the solver reports only the save grid, with the
numbers of accepted and rejected steps per save interval in `result.n_accepted`
and `result.n_rejected`.


## Tolerances

A step is accepted when its tolerance-weighted local error is at most one:

$$\mathrm{err} = \sqrt{\frac{1}{d} \sum_{i=1}^{d}
\left(\frac{D_i}{\mathrm{atol} + \mathrm{rtol}\,|x_i|}\right)^2} \;\le\; 1,$$

where $x_i$ is the predicted value of component $i$ and $D_i$ the standard deviation
that the step's (calibrated) process noise adds to it. The defaults are
`atol=1e-4` and `rtol=1e-2`; tighter tolerances give shorter steps and a more
accurate solution.

## Controllers

The controller turns the error of the step just taken into the next step size. Two
are built in and passed as `controller=`:

| Controller | Next step | Use when |
| --- | --- | --- |
| `PIController(order=q)` (default) | $h \cdot s \cdot \mathrm{err}^{-\alpha} \cdot (\mathrm{err}_\mathrm{prev} / \mathrm{err})^{\beta}$ | almost always: the last factor damps oscillation around the tolerance |
| `PController(order=q)` | $h \cdot s \cdot \mathrm{err}^{-\alpha}$ | you want memoryless, predictable behavior |

By default $s = 0.9$; $\alpha = 0.7/q$ and $\beta = 0.4/q$ for `PIController`
(Gustafsson, 1991) and $\alpha = 1/q$ for `PController`. The factor applied to $h$
is clipped to `[min_factor, max_factor] = [0.2, 5]`.

A custom controller only needs a `propose(h, err, err_prev)` method. The solver
calls it inside its compiled loop, so write it with `jax.numpy`; there a
non-positive `err_prev` means there is no previous error (the first step, or right
after a rejected one).

In [ ]:
result_p = gaussian_filter_adaptive(
    mu_0, P0_sqr, prior, measure, save_at, atol=1e-4, rtol=1e-3,
    controller=PController(order=2),
)

## Smoothing

By default the adaptive solver only filters. With `smoother=True` it also runs a
fixed-point smoother during the forward pass, which stores one backward step per
save interval however many steps it takes (Krämer, *Adaptive Probabilistic ODE
Solvers Without Adaptive Memory Requirements*, 2025). `rts_smoother` then applies
to the result. `smoother=True` cannot yet be combined with `obs_model`.

In [ ]:
result = gaussian_filter_adaptive(
    mu_0, P0_sqr, prior, measure, save_at, atol=1e-4, rtol=1e-3, smoother=True
)
m_smooth, P_smooth_sqr = rts_smoother(result)

## When a solve does not finish

The solver never raises inside a compiled solve. If a save interval needs more than
`max_steps` steps (4096 by default), it stops there and sets
`result.success = False`; the saves from that point on hold the last accepted
state. Raise `max_steps` or loosen the tolerances. `success` is also `False` when
the log-likelihood is not finite.

See also [Diffusion calibration](../calibration/), for the $\sigma^2$ that enters the
error estimate, and the [Calibration modes](../examples/calibration-modes/)
notebook.